# Notebook 60 — Behavioral Anomaly Detection

Demonstrates `multigen.behavioral`: behavioral fingerprinting, KL divergence drift
monitoring, latency anomaly detection, inter-agent communication pattern analysis,
and the `BehavioralAnomalyDetector` facade.  All cells are self-contained — no
external services or real LLM calls are required.

## 1. Setup

In [ ]:
import sys, importlib.util

sys.path.insert(0, '../sdk')

def load(name):
    spec = importlib.util.spec_from_file_location(
        f'multigen.{name}', f'../sdk/multigen/{name}.py')
    mod = importlib.util.module_from_spec(spec)
    sys.modules[f'multigen.{name}'] = mod
    spec.loader.exec_module(mod)
    return mod

beh = load('behavioral')

from multigen.behavioral import (
    BehavioralAnomalyDetector,
    BehavioralProfiler,
    KLDriftMonitor,
    LatencyAnomalyDetector,
    InterAgentCommunicationMonitor,
    BehavioralFingerprint,
    DriftResult,
    BehavioralDriftAlert,
    LatencyAnomaly,
    LatencyProfile,
    MessagePattern,
    BehavioralAnomalyReport,
)

print('multigen.behavioral loaded OK')
print('Exported symbols:', [
    'BehavioralAnomalyDetector', 'BehavioralProfiler', 'KLDriftMonitor',
    'LatencyAnomalyDetector', 'InterAgentCommunicationMonitor',
    'BehavioralFingerprint', 'DriftResult', 'BehavioralDriftAlert',
    'LatencyAnomaly', 'LatencyProfile', 'MessagePattern',
    'BehavioralAnomalyReport',
])

## 2. Behavioral Profiling

Feed 20 representative outputs to a `BehavioralProfiler`, build a fingerprint
for `agent1`, inspect the key statistics, and commit it as the **baseline**.

In [ ]:
profiler = BehavioralProfiler()

# 20 normal, moderately-sized business-style responses
normal_outputs = [
    "The quarterly revenue figures show a steady increase of five percent compared to last year.",
    "Our analysis indicates that customer retention has improved due to the new onboarding process.",
    "The marketing campaign delivered strong results with a measurable uplift in brand awareness.",
    "Supply chain disruptions have been mitigated by diversifying our vendor relationships.",
    "The engineering team completed the migration to the new infrastructure on schedule.",
    "Customer satisfaction scores rose to an all-time high of ninety-two percent this quarter.",
    "We recommend allocating additional budget to the product development roadmap for next year.",
    "The compliance audit found no material weaknesses in our internal control framework.",
    "Headcount grew by twelve percent following the successful acquisition of a regional competitor.",
    "Operational costs were reduced by eight percent through process automation initiatives.",
    "The new pricing strategy has been well received by enterprise customers in the segment.",
    "Research and development investment remains at six percent of total revenue this period.",
    "Partnership negotiations with three major distributors are progressing as planned.",
    "The data platform upgrade will enable real-time analytics for all business units.",
    "Employee engagement survey results indicate a high level of satisfaction with leadership.",
    "The product launch exceeded initial sales projections by twenty percent in the first month.",
    "Risk assessment of the new market entry identified three primary areas requiring attention.",
    "Legal review of the proposed contract amendments has been completed without major issues.",
    "The board approved the revised capital expenditure plan for the upcoming fiscal year.",
    "Integration of the acquired company is on track and expected to complete within six months.",
]

for text in normal_outputs:
    profiler.observe('agent1', text)

# Build and inspect the fingerprint
fp = profiler.build_fingerprint('agent1')

print(f'Agent ID       : {fp.agent_id}')
print(f'Sample count   : {fp.sample_count}')
print(f'Length mean    : {fp.response_length_mean:.1f} chars')
print(f'Length std     : {fp.response_length_std:.1f} chars')
print(f'Vocabulary size: {fp.vocabulary_size} unique tokens')
print(f'Avg sentences  : {fp.avg_sentence_count:.2f}')
print(f'Top bigrams    :')
for bigram, count in fp.top_bigrams[:5]:
    print(f'  "{bigram}" -> {count}')

# Commit as baseline
profiler.set_baseline('agent1')
print('\nBaseline committed for agent1.')

## 3. Fingerprint Drift Detection

Feed 20 more outputs with a **very different style** (short, terse, emoji-heavy
single-word responses) and call `profiler.compare("agent1")` to obtain a
`DriftResult` indicating which metric drifted and by how much.

In [ ]:
# Clear current samples so the new batch represents the post-drift window
# (re-create profiler, restore baseline from fingerprint, then add drifted samples)
profiler2 = BehavioralProfiler()

# First replay normal outputs and set baseline
for text in normal_outputs:
    profiler2.observe('agent1', text)
profiler2.set_baseline('agent1')

# Now clear the rolling buffer and feed extremely short, terse outputs
# BehavioralProfiler keeps last 100; we override by observing 100 new samples
drifted_outputs = [
    "ok",
    "done",
    "yes",
    "no",
    "k",
    "sure",
    "nope",
    "maybe",
    "fine",
    "alright",
    "yep",
    "nah",
    "hmm",
    "ok.",
    "gotcha",
    "ack",
    "noted",
    "sure thing",
    "on it",
    "will do",
]

# Flood the buffer so compare() sees only drifted samples (101 drifted vs 20 normal)
for text in drifted_outputs * 5:   # 100 drifted samples pushes out the 20 normal ones
    profiler2.observe('agent1', text)

# Compare current window against baseline
drift = profiler2.compare('agent1')

if drift is not None:
    print(f'DriftResult detected!')
    print(f'  Agent ID        : {drift.agent_id}')
    print(f'  Drifted metric  : {drift.metric}')
    print(f'  Baseline value  : {drift.baseline_value:.2f}')
    print(f'  Current value   : {drift.current_value:.2f}')
    print(f'  Drift magnitude : {drift.drift_magnitude:.2%} relative change')
else:
    print('No significant drift detected (unexpected).')

## 4. KL Divergence Drift Monitor

Set a baseline on 10 normal texts, then check 10 anomalous texts using a
completely different vocabulary.  Expect a `BehavioralDriftAlert` with a
non-trivial `kl_divergence` value.

In [ ]:
kl_monitor = KLDriftMonitor()

# Baseline: formal English business prose
baseline_texts = [
    "revenue increased significantly during the fiscal quarter results",
    "customer satisfaction metrics improved following the product update launch",
    "the board approved the strategic investment in infrastructure development",
    "operational efficiency gains were reported across all business units today",
    "the compliance team completed the annual regulatory audit without findings",
    "enterprise clients renewed contracts representing substantial recurring revenue",
    "the engineering department delivered the platform migration ahead of schedule",
    "market expansion strategy focuses on high-growth emerging regional markets",
    "the finance team finalized the annual budget planning and forecasting cycle",
    "human resources launched a new talent acquisition programme for senior roles",
]

kl_monitor.set_baseline('agent1', baseline_texts)
print('Baseline set for agent1 ({} texts).'.format(len(baseline_texts)))

# Anomalous texts: highly technical jargon from a completely different domain
anomalous_texts = [
    "mitochondria atp synthesis electron transport phosphorylation krebs",
    "supernova nebula quasar redshift parallax spectroscopy parsec magnitude",
    "fourier transform laplacian eigenvalue eigenvector divergence gradient curl",
    "tectonic subduction magma lithosphere mantle convection isostasy seismology",
    "polymer crystallinity tensile modulus viscoelastic rheology stress fracture",
    "dendrite synapse axon neurotransmitter receptor dopamine serotonin cortex",
    "encryption decryption cipher plaintext ciphertext hash nonce salt padding",
    "allele genotype phenotype mutation epistasis heterozygous diploid locus",
    "thermodynamics entropy enthalpy gibbs helmholtz boltzmann statistical",
    "stochastic markov chain stationary distribution ergodic recurrence transient",
]

alert = kl_monitor.check('agent1', anomalous_texts, threshold=0.5)

if alert is not None:
    print(f'\nBehavioralDriftAlert raised!')
    print(f'  Agent ID       : {alert.agent_id}')
    print(f'  KL Divergence  : {alert.kl_divergence:.4f}')
    print(f'  Threshold      : {alert.threshold}')
    import datetime
    ts = datetime.datetime.fromtimestamp(alert.timestamp).isoformat()
    print(f'  Timestamp      : {ts}')
else:
    print('No drift alert — KL divergence below threshold.')

## 5. Latency Anomaly Detection

Record 50 latencies clustered around 100 ms to establish a stable profile, then
inject a single 5000 ms outlier.  The detector should return a `LatencyAnomaly`
with a high z-score.

In [ ]:
import random
random.seed(42)

lat_detector = LatencyAnomalyDetector()

# Record 50 normal latencies (Gaussian around 100 ms, std=10 ms)
anomaly = None
for _ in range(50):
    ms = random.gauss(100, 10)
    result = lat_detector.record('agent1', ms)
    if result is not None:
        anomaly = result  # should not trigger on normal traffic

profile = lat_detector.get_profile('agent1')
print('Latency profile after 50 normal observations:')
print(f'  mean_ms       : {profile.mean_ms:.2f} ms')
print(f'  std_ms        : {profile.std_ms:.2f} ms')
print(f'  p95_ms        : {profile.p95_ms:.2f} ms')
print(f'  p99_ms        : {profile.p99_ms:.2f} ms')
print(f'  sample_count  : {profile.sample_count}')
print(f'  Anomaly during normal phase: {anomaly}\n')

# Inject a severe outlier: 5000 ms
outlier_ms = 5000.0
anomaly = lat_detector.record('agent1', outlier_ms)

if anomaly is not None:
    print(f'LatencyAnomaly detected!')
    print(f'  Agent ID    : {anomaly.agent_id}')
    print(f'  Latency     : {anomaly.latency_ms:.1f} ms')
    print(f'  Z-score     : {anomaly.z_score:.2f}  (threshold = 3.0)')
    import datetime
    ts = datetime.datetime.fromtimestamp(anomaly.timestamp).isoformat()
    print(f'  Timestamp   : {ts}')
else:
    print('No anomaly detected (unexpected).')

## 6. Inter-Agent Communication Monitor

Record message patterns between 4 agents, inspect `get_patterns()`, then call
`detect_anomalies()` to surface three distinct anomaly types:

- **Self-messaging** — an agent sends a message to itself.
- **Communication loop** — agent A sends to B and B sends back to A.
- **Excessive volume** — more than 100 messages on a single channel.

In [ ]:
comm_monitor = InterAgentCommunicationMonitor()

# Normal agent-to-agent messages
comm_monitor.record_message('agent1', 'agent2', 'Here is the analysis report for review.')
comm_monitor.record_message('agent1', 'agent3', 'Please validate these findings before submission.')
comm_monitor.record_message('agent2', 'agent4', 'Forwarding processed data to the storage agent.')
comm_monitor.record_message('agent3', 'agent4', 'Sending compliance-checked output downstream.')

# Anomaly 1: self-messaging — agent2 sends a message to itself
comm_monitor.record_message('agent2', 'agent2', 'Echoing internal state for debugging purposes.')

# Anomaly 2: communication loop — agent1 -> agent3 -> agent1 (bidirectional)
comm_monitor.record_message('agent3', 'agent1', 'Sending response back to originating agent.')

# Anomaly 3: excessive volume — agent1 floods agent4 with 105 messages
for i in range(105):
    comm_monitor.record_message('agent1', 'agent4', f'Batch message {i}: status update payload.')

# Show all observed communication patterns
patterns = comm_monitor.get_patterns()
print(f'Total communication patterns observed: {len(patterns)}\n')
print(f'{"From":<12} {"To":<12} {"Count":>7}  {"Avg Len":>9}')
print('-' * 46)
for p in sorted(patterns, key=lambda x: x.message_count, reverse=True):
    print(f'{p.from_agent:<12} {p.to_agent:<12} {p.message_count:>7}  {p.avg_message_length:>9.1f}')

# Detect anomalies
anomalies = comm_monitor.detect_anomalies(threshold_messages=100)
print(f'\nAnomalies detected ({len(anomalies)}):')
for i, desc in enumerate(anomalies, 1):
    print(f'  [{i}] {desc}')

## 7. Full Anomaly Report

Use the `BehavioralAnomalyDetector` facade to wire together profiling, KL
monitoring, latency detection, and communication monitoring.  Call
`detector.full_report(["agent1", "agent2", "agent3"])` to obtain a consolidated
`BehavioralAnomalyReport` with a computed `risk_score`.

In [ ]:
import datetime

detector = BehavioralAnomalyDetector()

# --- Behavioral profiling: agent1 drifts, agent2 and agent3 stay stable ---

# agent1 baseline (normal business prose)
for text in normal_outputs:   # the 20 outputs defined in Section 2
    detector.profiler.observe('agent1', text)
detector.profiler.set_baseline('agent1')

# agent1 post-drift: flood with terse responses so fingerprint changes
for text in drifted_outputs * 5:
    detector.profiler.observe('agent1', text)

# agent2 — set baseline and keep stable (no additional observations)
for text in normal_outputs[:10]:
    detector.profiler.observe('agent2', text)
detector.profiler.set_baseline('agent2')

# agent3 — set baseline and keep stable
for text in normal_outputs[10:]:
    detector.profiler.observe('agent3', text)
detector.profiler.set_baseline('agent3')

# --- KL monitor baselines ---
detector.kl_monitor.set_baseline('agent1', baseline_texts)
detector.kl_monitor.set_baseline('agent2', baseline_texts[:5])
detector.kl_monitor.set_baseline('agent3', baseline_texts[5:])

# --- Latency: agent2 has an anomalous spike ---
for _ in range(50):
    detector.latency.record('agent2', random.gauss(80, 8))
lat_anomaly = detector.latency.record('agent2', 4500.0)

# --- Communication anomalies ---
# Self-message on agent3
detector.comm_monitor.record_message('agent3', 'agent3', 'Internal loop message.')
# Bidirectional loop between agent1 and agent2
detector.comm_monitor.record_message('agent1', 'agent2', 'Request to agent2.')
detector.comm_monitor.record_message('agent2', 'agent1', 'Response back to agent1.')
# Excessive volume: agent1 -> agent3
for i in range(110):
    detector.comm_monitor.record_message('agent1', 'agent3', f'High-frequency ping {i}')

# --- Generate full report ---
report = detector.full_report(['agent1', 'agent2', 'agent3'])

print('=' * 60)
print('BehavioralAnomalyReport')
print('=' * 60)
print(f'Report ID      : {report.report_id}')
ts_str = datetime.datetime.fromtimestamp(report.timestamp).isoformat()
print(f'Timestamp      : {ts_str}')
print(f'Risk Score     : {report.risk_score:.3f}  (range 0.0 – 1.0)')

print(f'\nDrift Alerts ({len(report.drift_alerts)}):')
for dr in report.drift_alerts:
    print(f'  agent={dr.agent_id}  metric={dr.metric}')
    print(f'    baseline={dr.baseline_value:.2f}  current={dr.current_value:.2f}  magnitude={dr.drift_magnitude:.2%}')

if lat_anomaly:
    print(f'\nLatency Anomaly (event-driven, outside report):')
    print(f'  agent={lat_anomaly.agent_id}  latency={lat_anomaly.latency_ms:.1f}ms  z={lat_anomaly.z_score:.2f}')

print(f'\nCommunication Anomalies ({len(report.communication_anomalies)}):')
for i, desc in enumerate(report.communication_anomalies, 1):
    print(f'  [{i}] {desc}')

print('=' * 60)
risk_label = 'LOW' if report.risk_score < 0.33 else ('MEDIUM' if report.risk_score < 0.66 else 'HIGH')
print(f'Overall Risk Level: {risk_label} ({report.risk_score:.3f})')